# **Modelo LightGCN**
### Proyecto Hito 2
### Sistemas Recomendadores IIC3633-1 2025-2
### **Grupo 3:** 

- Nicolás Antonio Bueno Abett de la Torre 

- Felipe Andrés Fuentes González

- Jorge Andrés Jacque Palma

- Francisco Nicolás Solís Gormaz

## Índice

>[0- Instalación de librerías](#0--instalación-de-librerías)

>[1- Carga de datos](#1--carga-de-datos)

>[2- Definición del modelo, formateo de datos, y entrenamiento](#2--definición-del-modelo-formateo-de-datos-y-entrenamiento)

>[3- Generación de recomendaciones](#3--generación-de-recomendaciones)

>[4- Métricas](#4--métricas)

>[5- Referencias](#5--referencias)

## 0- Instalación de librerías

En caso de usar Colab, correr estas celdas. Si se corre en local, se deben tener exactamente las mismas versiones de las librerías indicadas en estas celdas. Las versiones de las librerías son:

- numpy: 1.25.0
- pandas: 2.2.2
- scipy: 1.10.1
- tqdm: 4.66.6
- torch: 2.5.1+cpu
- recbole: 1.2.1

In [ ]:
# !pip uninstall -y numpy
# !pip install numpy==1.25

In [ ]:
# !pip uninstall -y pandas
# !pip install numpy==2.2.2

In [ ]:
# !pip uninstall -y scipy
# !pip install scipy==1.10.1

In [ ]:
# !pip uninstall -y tqdm
# !pip install tqdm==4.66.6

In [ ]:
# !pip uninstall -y torch
# !pip uninstall -y torchvision
# !pip uninstall -y torchaudio
# !pip install torch==2.5.1+cpu --index-url https://download.pytorch.org/whl/cpu

In [ ]:
# !pip install recbole==1.2.1

## 1- Carga de datos

In [4]:
import pandas as pd

Se leen los archivos de datos y se almacenan en un dataframe. El primero, df_original, lee el archivo original del dataset. El segundo, "df_con_ids", lee el archivo del dataset modificado con ids de usuario e ítem agregadas, tal como se hizo en el Hito 1 del proyecto al implementar los modelos referenciales (es el mismo archivo creado en /modelos_ref/modelos_ref.ipynb). El tercero, "df_final", es el mismo contenido de "df_con_ids", pero tomando sólo las columnas de id de usuario, id de ítem, y rating, con estos últimos convertidos a escala del 1 al 5, tal como se hizo en el Hito 1 del proyecto al implementar los modelos referenciales (es el mismo archivo creado en /modelos_ref/modelos_ref.ipynb). Los dataframes son:

In [5]:
df_original = pd.read_csv('video_game_reviews.csv')
df_con_ids = pd.read_csv('video_game_reviews_with_userid.csv')
df_final = pd.read_csv('video_game_reviews_with_userid_clean.csv')

En este diccionario "info_videojuegos" se guarda la id del videojuego junto a su título correspondiente, para luego obtener información de este (como su género, plataforma, etc.):

In [6]:
info_videojuegos = dict(zip(df_con_ids['item_id'], df_con_ids['Game Title']))

El dataframe final que se utilizará para las recomendaciones es:

In [7]:
df_final

,user_id,item_id,rating
0,861,12,3.670051
1,1295,38,3.862944
2,1131,21,2.695431
3,1096,4,3.873096
4,1639,14,3.030457
...,...,...,...
47769,1293,21,4.197970
47770,2485,37,2.431472
47771,2675,3,2.685279
47772,2599,37,2.258883


## 2- Definición del modelo, formateo de datos, y entrenamiento

Se define formatean los datos para la librería RecBole, se define el modelo, y se entrena:

In [ ]:
import os
import pandas as pd
import torch
from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.model.general_recommender import LightGCN
from recbole.trainer import Trainer
from recbole.utils.case_study import full_sort_topk

# === 1. Cargar CSV y renombrar columnas según RecBole ===
df_lightgcn = pd.read_csv("video_game_reviews_with_userid_clean.csv")
df_lightgcn.columns = [
    "user_id:token",
    "item_id:token",
    "rating:float"
]

# === 2. Guardar dataset en la carpeta mínima para RecBole ===
dataset_name = "video_game_reviews_recbole"
dataset_dir = os.path.join(".", dataset_name)
os.makedirs(dataset_dir, exist_ok=True)
inter_file = os.path.join(dataset_dir, f"{dataset_name}.inter")
df_lightgcn.to_csv(inter_file, sep="\t", index=False)

# === 3. Configuración ===
config_dict = {
    "data_path": ".",
    "dataset": dataset_name,
    "dataset_file": {"inter": f"{dataset_name}/{dataset_name}.inter"},
    "field_separator": "\t",
    "USER_ID_FIELD": "user_id",
    "ITEM_ID_FIELD": "item_id",
    "RATING_FIELD": "rating",
    "load_col": {"inter": ["user_id", "item_id", "rating"]},
    "field_type": {
        "user_id": "token",
        "item_id": "token",
        "rating": "float"
    },
    "eval_args": {"split": {"RS": [0.8, 0.1, 0.1]}, "mode": "full"},
    "epochs": 10,
    "train_batch_size": 2048,
    "eval_batch_size": 4096,
    "embedding_size": 64,
    "show_progress": True,
    "learning_rate": 0.001,
    "reg_weight": 1e-5,
    "n_layers": 3,
    "topk": [10],
    "device": "cpu"  # o "cuda" si tienes GPU compatible
}

# === 4. Crear config, dataset y dataloaders ===
config = Config(model='LightGCN', dataset=dataset_name, config_dict=config_dict)
dataset = create_dataset(config)
train_data, valid_data, test_data = data_preparation(config, dataset)

# === 5. Crear modelo y trainer ===
model = LightGCN(config, train_data.dataset).to(config['device'])
trainer = Trainer(config, model)

# === 6. Entrenar ===
trainer.fit(train_data, valid_data)
print()
print("Entrenamiento finalizado")

c:\Users\felip\AppData\Local\Programs\Python\Python311\Lib\site-packages\recbole\data\dataset\dataset.py:648: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  feat[field].fillna(value=0, inplace=True)
c:\Users\felip\AppData\Local\Programs\Python\Python311\Lib\site-packages\recbole\data\dataset\dataset.py:650: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the interm

Entrenamiento finalizado


## 3- Generación de recomendaciones

Ahora, a partir del entrenamiento anterior se obtienen las listas de recomendación top 10 para cada usuario. Se comienza usando las ids internas de la librería, para luego convertirlas a las originales del dataset:

In [ ]:
# === 7. Generar recomendaciones Top-K para todos los usuarios ===
# Solo usuarios que existen en test_data
dataset_obj = test_data.dataset
# internal IDs de usuarios en test
uid_series = [uid for uid in range(dataset_obj.user_num) 
              if test_data.uid2history_item[uid] is not None]

topk = 10
topk_result = full_sort_topk(uid_series, model, test_data, k=topk, device=config['device'])

# topk_result es torch.return_types.topk
topk_indices = topk_result.indices  # IDs internos de items
topk_scores = topk_result.values    # Scores

# === 8. Convertir a ids originales (más legible) ===
print("Recomendaciones ejemplo:")
for uid in range(5):
    # Convertir la fila de indices a lista de enteros
    item_indices = topk_indices[uid].tolist()
    item_tokens = [dataset.id2token(dataset.iid_field, int(iid)) for iid in item_indices]
    user_token = dataset.id2token(dataset.uid_field, uid)
    print(f"Usuario {user_token} → Ítem {item_tokens}")

Recomendaciones ejemplo:
Usuario [PAD] → Ítem ['33', '21', '40', '3', '25', '4', '13', '37', '28', '1']
Usuario 861 → Ítem ['36', '33', '8', '12', '20', '26', '16', '32', '34', '3']
Usuario 1295 → Ítem ['6', '38', '35', '31', '39', '15', '13', '16', '28', '17']
Usuario 1131 → Ítem ['29', '16', '30', '1', '9', '31', '28', '35', '14', '5']
Usuario 1096 → Ítem ['36', '33', '16', '3', '20', '25', '22', '9', '30', '8']


Se guardan en un diccionario las recomendaciones con las ids de usuario e ítem originales, convirtiendo las internas de RecBole a las originales del dataset. El diccionario tiene como llaves la id del usuario y su valor es una lista con las 10 id de ítems recomendados: 

In [13]:
# Convertir IDs internos de usuarios a originales
user_original_ids = dataset_obj.id2token('user_id', uid_series)

# Convertir IDs internos de items a originales
topk_items_original = [
    dataset_obj.id2token('item_id', topk_result.indices[i])
    for i in range(len(uid_series))
]

# Crear diccionario final
recommendations_dict = {user_id: items for user_id, items in zip(user_original_ids, topk_items_original)}

# Ejemplo
for user_id, items in list(recommendations_dict.items())[:5]:
    print(f"Usuario {user_id} → {items}")

Usuario 861 → ['33' '21' '40' '3' '25' '4' '13' '37' '28' '1']
Usuario 1295 → ['36' '33' '8' '12' '20' '26' '16' '32' '34' '3']
Usuario 1131 → ['6' '38' '35' '31' '39' '15' '13' '16' '28' '17']
Usuario 1096 → ['29' '16' '30' '1' '9' '31' '28' '35' '14' '5']
Usuario 1639 → ['36' '33' '16' '3' '20' '25' '22' '9' '30' '8']


Usuarios con recomendaciones (en total el dataset tiene 3000):

In [14]:
len(recommendations_dict)

3000

## 4- Métricas

La ejecución del modelo LightGCN de la librería RecBole entrega de por sí ciertas métricas: Recall@K, Precision@K, MRR@K, NDCG@K, y HitScore@K. Estas métricas se obtienen primero del objeto Trainer de la librería y se calculan sobre la data de testeo. Se obtiene un tipo OrderedDict que se convierte a diccionario:

In [15]:
test_result = trainer.evaluate(test_data)
metricas_finales = dict(test_result)
print(metricas_finales)

{'recall@10': 0.5472, 'mrr@10': 0.3436, 'ndcg@10': 0.3734, 'hit@10': 0.5937, 'precision@10': 0.0658}


c:\Users\felip\AppData\Local\Programs\Python\Python311\Lib\site-packages\recbole\trainer\trainer.py:583: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.loa

Ahora, de las anteriores se guardan en variables sólo las métricas relevantes (las que se utilizan en este proyecto, todas excepto MRR@K) para finalmente mostrarlas en un resumen:

In [16]:
recall_lightgcn = metricas_finales['recall@10']
precision_lightgcn = metricas_finales['precision@10']
ndcg_lightgcn = metricas_finales['ndcg@10']
hitscore_lightgcn = metricas_finales['hit@10']

print(f"Recall@10: {recall_lightgcn}")
print(f"Precision@10: {precision_lightgcn}")
print(f"NDCG@10: {ndcg_lightgcn}")
print(f"HitScore@10: {hitscore_lightgcn}")

Recall@10: 0.5472
Precision@10: 0.0658
NDCG@10: 0.3734
HitScore@10: 0.5937


Otra métrica que se utilizará es el F1 Score. Se calcula y guarda en una variable:

In [17]:
f1score_lightgcn = 2 * (precision_lightgcn * recall_lightgcn) / (precision_lightgcn + recall_lightgcn)
print(f"F1 Score@10: {f1score_lightgcn}")

F1 Score@10: 0.11747393148450244


Ahora, se calcula el MAP@K:

## 5- Referencias

- [1] Documentación de RecBole: https://github.com/RUCAIBox/RecBole?utm_source=chatgpt.com

## 6- Anexo: Intento inicial de código (no funciona)

In [ ]:
import os
import pandas as pd
from recbole.quick_start import run_recbole
from recbole.utils.case_study import full_sort_topk

# === 1. Cargar tu dataset CSV ===
df_lightgcn = pd.read_csv("video_game_reviews_with_userid_clean.csv")

# === 2. Crear estructura que RecBole necesita ===

# # RecBole requiere un campo de tiempo, así que añadimos uno aunque LightGCN no lo use
# df_lightgcn["timestamp"] = pd.RangeIndex(start=1, stop=len(df_lightgcn) + 1)

df_lightgcn.columns = [
    "user_id:token",
    "item_id:token",
    "rating:float"
]

dataset_name = "video_game_reviews_recbole"
dataset_dir = os.path.join(".", dataset_name)  # carpeta mínima en el directorio actual

# Crear la carpeta si no existe
os.makedirs(dataset_dir, exist_ok=True)

# Nombre del archivo .inter
inter_file = os.path.join(dataset_dir, f"{dataset_name}.inter")

# Guardar el CSV en la carpeta del dataset
df_lightgcn.to_csv(inter_file, sep="\t", index=False)

# === 3. Configuración para LightGCN ===
config_dict = {
    "data_path": ".",
    "dataset": "video_game_reviews_recbole",
    "dataset_file": {"inter": "video_game_reviews_recbole/video_game_reviews_recbole.inter"},

    "field_separator": "\t",
    "USER_ID_FIELD": "user_id",
    "ITEM_ID_FIELD": "item_id",
    "RATING_FIELD": "rating",
    "load_col": {"inter": ["user_id", "item_id", "rating"]},

    "field_type": {
    "user_id": "token",
    "item_id": "token",
    "rating": "float"
    },

    # División de datos 80/10/10 (train/valid/test)
    "eval_args": {"split": {"RS": [0.8, 0.1, 0.1]}, "mode": "full"},
    "epochs": 10,                    # puedes aumentar para mejor entrenamiento
    "train_batch_size": 2048,
    "eval_batch_size": 4096,
    "embedding_size": 64,
    "show_progress": True,

    # LightGCN hiperparámetros (opcional ajustar)
    "learning_rate": 0.001,
    "reg_weight": 1e-5,
    "n_layers": 3,
    "topk": [10],

    #Evitar creación de carpeta /log y registro de checkpoints:
    # 'show_progress': True,
    # 'save_model': False,        # ❌ No guardar checkpoints
    # 'save_dataset': False,      # ❌ No guardar dataset cache
    # 'save_reproduce': False,    # ❌ No guardar archivos de reproducción
    # 'logger': None,  
}

# === 4. Entrenar LightGCN ===
config, model, dataset, trainer = run_recbole(
    model="LightGCN",
    dataset='video_game_reviews_recbole',
    config_dict=config_dict
)

print("\n Entrenamiento finalizado correctamente.")

# === 5. Generar recomendaciones ===
topk = 10
topk_result = full_sort_topk(model, dataset, k=topk, device=model.device)

# === 6. Convertir a IDs originales (más legible) ===
if isinstance(topk_result, tuple):
    topk_items, topk_scores = topk_result
else:
    topk_items = topk_result

# Mostrar ejemplo: top 5 usuarios con sus 10 recomendaciones
print("\n🎮 Recomendaciones ejemplo:")
user_ids = list(range(5))
for uid in user_ids:
    tokens = [dataset.id2token(dataset.iid_field, iid) for iid in topk_items[uid]]
    print(f"Usuario {dataset.id2token(dataset.uid_field, uid)} → {tokens}")